In [ ]:
# test-thresholds-&-different-analyses.ipynb
# this script will output following parameters for a set of vlgut1-psd95 staining:
# coloc of channels: pearsons corr coef, pearsons corr coef - in rot cond, pearsons p value, pearsons p value - in rot cond, 
# manders overlap coef, manders overlap coef - in rot cond, overlap (um2), overlap (um2) - in rot cond,
# for each channel separately: vglut1_threshold, psd95_threshold, vglut MFI, psd95 MFI, vglut1 staining area (um2), psd95 staining area (um2)

# required packages
import czifile # to import a .czi file
from microfilm.microplot import microshow
import numpy as np
import matplotlib.pyplot as plt
from skimage.restoration import rolling_ball
from skimage.exposure import equalize_adapthist
from skimage import measure
from skimage.transform import rotate
from skimage import morphology
from skimage import filters
from skimage.filters import gaussian, try_all_threshold
from lxml import etree # required library to load the metadata
import pandas as pd

In [ ]:
path = "/Volumes/KINGSTON/code/phd/image-analysis/synapse-counting/test-images-VLGUT1-PSD95-A/OE_Exp1_IHC_Exp1_HA-GPR37L1_555-VGLUT1_647-PSD95_63X_airyscan_1.8zoom_CA1_SO.czi"
image = czifile.imread(path)
image.shape

In [ ]:
# this retrieves the metadata and the extracts to pixel_to_um conversion

czi = czifile.CziFile(path)
czi_xml_str = czi.metadata() # gets the metadata in a xml string format
czi_parsed = etree.fromstring(czi_xml_str) # parses the czi_xml_str file

# finds the strings 
size_x = czi_parsed.find(".//SizeX")
size_y = czi_parsed.find(".//SizeY")
scaling_x = czi_parsed.find(".//ScalingX")
scaling_y = czi_parsed.find(".//ScalingY")

# extracting the required values to calculate the 
size_x_value = int(size_x.text)
size_y_value = int(size_y.text)
scaling_x_value = float(scaling_x.text)
scaling_y_value = float(scaling_y.text)

# calculater the pixel to micrometer (um)
# first checking whether the X and Y dimensions of the image are equal
if size_x_value == size_y_value and scaling_x_value == scaling_y_value:
    pixel_size = ((scaling_x_value*1000000000)/size_x_value) # conversion from meter to micrometer
    
print(pixel_size, "um per pixel")
print(size_y_value, size_y_value)

In [ ]:
# getting rid of all the extra channels, splitting the channels and showing them one by one.
image_squeezed = np.squeeze(image)
print(image_squeezed.shape)
vglut1 = image_squeezed[0,:,:]
psd95 = image_squeezed[1,:,:]

fig, axs = plt.subplots(1, 2, figsize = (30, 30))
microshow(vglut1, ax=axs[0], label_text = 'VLGUT1')
microshow(psd95, ax=axs[1], label_text = 'PSD95')

In [ ]:
# now a preproccesing step, removal of background with rolling ball radius of 10x
background_vglut1 = rolling_ball(vglut1, radius = 10)
background_psd95 = rolling_ball(psd95, radius = 10)
vglut1_bs = vglut1 - background_vglut1
psd95_bs = psd95 - background_psd95

fig, axs = plt.subplots(1, 2, figsize = (30, 30))
microshow(vglut1_bs, ax=axs[0], label_text = 'VLGUT1')
microshow(psd95_bs, ax=axs[1], label_text = 'PSD95')

In [ ]:
# trying out CLAHE

vglut1_bs_clahe = equalize_adapthist(vglut1_bs, clip_limit=0.005, kernel_size = 150, nbins = 265)

fig, axs = plt.subplots(1, 2, figsize = (30, 30))
microshow(vglut1_bs_clahe, ax=axs[0], label_text = 'PSD95 CLAHE')
microshow(vglut1_bs, ax=axs[1], label_text = 'PSD95')


In [ ]:
# trying out the the tophat filter
footprint = morphology.disk(5)
vglut1_bs_toph = morphology.white_tophat(vglut1_bs_clahe, footprint)

fig, axs = plt.subplots(1, 2, figsize = (30, 30))
microshow(vglut1_bs_toph, ax=axs[0], label_text = 'PSD95 tophat')
microshow(vglut1_bs_clahe, ax=axs[1], label_text = 'PSD95')

In [ ]:
# for the thresholding, it is advised to perform a gaussian blur (in this case a light one with sigma 1)
psd95_pre = gaussian(psd95_bs, sigma=1, preserve_range=True)

vglut1_pre = gaussian(vglut1_bs, sigma=1, preserve_range=True)

fig, axs = plt.subplots(2, 2, figsize = (30, 30))
microshow(psd95_bs, ax=axs[0,0])
microshow(psd95_pre, ax=axs[0,1])
microshow(vglut1_bs, ax=axs[1,0])
microshow(vglut1_pre, ax=axs[1,1])

In [ ]:
#######################################################################
# pearsons correlation of colocalization - based on pixel intensities #
#######################################################################

# control (vglut1-psd95_rot)
psd95_pre_rot = rotate(psd95_pre, 90)
pcc_rot, pval_rot = measure.pearson_corr_coeff(vglut1_pre, psd95_pre_rot)
print(f"PCC_rot: {pcc_rot}, p-val_rot: {pval_rot}")
# actual (vglut1-psd95)
pcc, pval = measure.pearson_corr_coeff(vglut1_pre, psd95_pre)
print(f"PCC: {pcc}, p-val: {pval}")

In [ ]:
# checking the different threshold algorithms
fig, ax = try_all_threshold(vglut1_pre, figsize=(20, 16), verbose=False)
plt.show()

In [ ]:
#########################################################
# Mander's overlap coefficient - based on binary images #
#########################################################

# rotating the psd95 image as control
psd95_pre_rot = rotate(psd95_pre, 90)

# getting the threshold (otsu) of each image
vglut1_threshold = filters.threshold_otsu(vglut1_pre)
psd95_threshold = filters.threshold_otsu(psd95_pre)
psd95_threshold_rot = filters.threshold_otsu(psd95_pre_rot)

# applying the threhold
vglut1_pre_thr = vglut1_pre >= vglut1_threshold
psd95_pre_thr = psd95_pre >= psd95_threshold
psd95_pre_rot_thr = psd95_pre_rot >= psd95_threshold_rot

# measuring the overlap coeff
overlap_coeff = measure.manders_overlap_coeff(vglut1_pre_thr, psd95_pre_thr)
overlap_coeff_rot = measure.manders_overlap_coeff(vglut1_pre_thr, psd95_pre_rot_thr)



# printing out the measurements
print(f"Manders overlap coefficient: {overlap_coeff}, and for rotated control: {overlap_coeff_rot}")
print(f"Thresholds for vglut1: {vglut1_threshold}, and psd95: {psd95_threshold}")

# showing the thresholded images and the overlap between the image and rotated images
fig, axs = plt.subplots(1, 2, figsize = (40, 40))
microshow(vglut1_pre_thr, ax=axs[0], label_text = 'vglut1_pre_thr')
microshow(psd95_pre_thr, ax=axs[1], label_text = 'psd95_pre_thr')

In [ ]:
###################################
# vglut1 and psd95 overlap - in um #
###################################

# getting the overlap and nr of pixel that overlapped
overlap = vglut1_pre_thr & psd95_pre_thr
overlap_rot = vglut1_pre_thr & psd95_pre_rot_thr

overlap_pix = np.sum(overlap)
overlap_pix_rot = np.sum(overlap_rot)

overlap_um = overlap_pix * pixel_size * pixel_size
overlap_um_rot = overlap_pix_rot * pixel_size * pixel_size

print(f"overlap: {overlap_um} um2, overlap rotated control: {overlap_um_rot} um2")

In [ ]:
#####################################################################
# synaptic markers separately: MFI, puncta density, staining area #
#####################################################################

# mean fluorescence intensity (MFI) - based on intensities
vglut1_mfi = np.mean(vglut1_pre)
psd95_mfi = np.mean(psd95_pre)

# puncta density - based on binary images
vglut1_labeled = measure.label(vglut1_pre > vglut1_threshold, connectivity = 1) # 2 for 2d image
vglut1_prop = measure.regionprops_table(vglut1_labeled, properties = ["area"]) # only calculates one property, is faster
df_vglut1_prop = pd.DataFrame(vglut1_prop)
vglut1_puncta_nr = df_vglut1_prop.shape[0]

psd95_labeled = measure.label(psd95_pre > psd95_threshold, connectivity = 2)
psd95_prop = measure.regionprops_table(psd95_labeled, properties = ["area"])
df_pdf95_prop = pd.DataFrame(psd95_prop)
psd95_puncta_nr = df_pdf95_prop.shape[0]

fig, axs = plt.subplots(2, 2, figsize = (30, 30))
microshow(vglut1_labeled, ax=axs[0, 0])
microshow(vglut1_pre, ax=axs[0, 1])
microshow(psd95_pre, ax=axs[1, 0])
microshow(psd95_labeled, ax=axs[1, 1])

if image_squeezed.shape[1] == image_squeezed.shape[2]: # checking the image a square
    image_area = image_squeezed.shape[1]
    
    vglut1_puncta_density_per_100_um2 = (vglut1_puncta_nr / (pixel_size * size_y_value)**2) * 100
    psd95_puncta_density_per_100_um2 = (psd95_puncta_nr / (pixel_size * size_y_value)**2) * 100

print(image_area, pixel_size, size_x_value)

# staining area - based on binary images
vglut1_area_pix = np.sum(vglut1_pre_thr)
psd95_area_pix = np.sum(psd95_pre_thr)

vglut1_area_um2 = vglut1_area_pix * pixel_size * pixel_size
psd95_area_um2 = psd95_area_pix * pixel_size * pixel_size

# printing out the resultss
print(f"MFI of vglut1: {vglut1_mfi}, MFI of psd95: {psd95_mfi}")
print(f"Puncta nr of vglut1 : {vglut1_puncta_nr} and of psd95: {psd95_puncta_nr}")
print(f"Puncta density per 100 um2 of vglut1: {vglut1_puncta_density_per_100_um2}, and of psd95: {psd95_puncta_density_per_100_um2}")
print(f"Staining area of vlgut1: {vglut1_area_um2} um2 , and psd95: {psd95_area_um2} um2")

In [ ]:
df_vglut1_prop.head(10)

In [ ]:
# getting the data into a dataframe and writing out the data into a csv

import os
import pandas as pd

# get the filename
name_of_file = os.path.splitext(os.path.basename(path))[0]
split_filename = name_of_file.split("_")

# get only the experimental parameters from the filename
index_nums = [0, 1, 2, 3, 10, 11] # the indexes of the elements that I would like to extract fro; the filename
desired_parts = [split_filename[val] for val in index_nums]
desired_filename = "_".join(desired_parts)

# making the df to output
d = {"image_name": desired_filename, 
     "pearsons corr coef": pcc,
     "pearsons corr coef - in rot cond": pcc_rot,
     "pearsons p value": pval,
     "pearsons p value - in rot cond": pval_rot,
     "manders overlap coef": overlap_coeff, 
     "manders overlap coef - in rot cond": overlap_coeff_rot,
     "vglut1_threshold": vglut1_threshold,
     "psd95_threshold": psd95_threshold,
     "overlap (um2)": overlap_um,
     "overlap (um2) - in rot cond": overlap_um_rot,
     "vglut MFI": vglut1_mfi,
     "psd95 MFI": psd95_mfi,
     "vglut1 staining area (um2)": vglut1_area_um2,
     "psd95 staining area (um2)": psd95_area_um2
     }
df = pd.DataFrame.from_dict([d]).set_index("image_name")

# Writing the dataframe in a csv format into the a specific folder
output_filename = "threshold_values_" + desired_filename + ".csv"
output_path = "/mnt/d/code/phd/image-analysis/synapse-counting/output_data/" + output_filename
df.to_csv(output_path)